In [1]:
from sage.all import load, var, solve
import sys
import re
from collections import defaultdict
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
from scipy.stats import ranksums 
import time
from scipy.optimize import minimize
import math
from scipy.stats import poisson
import operator

In [2]:
sys.path.append('../package/')

In [3]:
import tau

In [4]:
sample = '0bfd1043-7346-fdd0-e050-11ac0c484cab'
snv_file = f'{sample}.consensus.20160830.somatic.snv_mnv.vcf.gz'
cnv_file = f'{sample}.consensus.20170119.somatic.cna.txt'
purity_file = 'consensus.20170217.purity.ploidy.txt'
output_dir = 'outputs'
output_file = os.path.join(output_dir, f'{sample}_trim.txt')

In [5]:
trim_df = tau.trim(sample, snv_file, cnv_file, purity_file, output_file)

Processing sample: 0bfd1043-7346-fdd0-e050-11ac0c484cab
SNV file: 0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20160830.somatic.snv_mnv.vcf.gz
CNV file: 0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20170119.somatic.cna.txt
Purity file: consensus.20170217.purity.ploidy.txt
Output file: outputs/0bfd1043-7346-fdd0-e050-11ac0c484cab_trim.txt


[W::bcf_hrec_check] Invalid tag name: "1000genomes_AF"
[W::bcf_hrec_check] Invalid tag name: "1000genomes_ID"
[W::bcf_hdr_register_hrec] The definition of Flag "INFO/dbsnp_somatic" is invalid, forcing Number=0
[E::idx_find_and_load] Could not retrieve index file for '0bfd1043-7346-fdd0-e050-11ac0c484cab.consensus.20160830.somatic.snv_mnv.vcf.gz'
[W::vcf_parse] Contig '1' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '10' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '11' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '12' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '13' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '14' is not defined in the header. (Quick workaround: index the file with tabix.)
[W::vcf_parse] Contig '15' is

Trim step completed successfully. Output saved to outputs/0bfd1043-7346-fdd0-e050-11ac0c484cab_trim.txt


/n/data1/hms/dbmi/park/jbrew/tools/miniforge3/envs/sage-env/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
categorize_df = tau.categorize(sample, df=trim_df)

In [7]:
revise_df, subclonal_CCF_df = tau.revise(sample, df=categorize_df)

Clustering subclones (Step 1)
Re-clustering subclones (Step 3)
Final subclone clustering (Step 4)


In [8]:
normalize_df = tau.normalize(sample, df=revise_df)

In [9]:
exposure_file = 'MuSiCal_assignment_PCAWG.txt'
signature_file = 'COSMIC_v3p2_SBS_WGS_MuSiCal.csv'

In [10]:
def ensure_signature_order(signatures):
    """Reorder the signatures DataFrame to match the standard 96 trinucleotide mutation types."""
    expected_types = generate_96_types()  # Uppercase types
    if not all(typ in signatures['Type'].values for typ in expected_types):
        raise ValueError("Signature file is missing some standard 96 trinucleotide types.")

    # Reorder signatures to match expected order
    signatures = signatures.set_index('Type').reindex(expected_types).reset_index()
    return signatures

def generate_96_types():
    """Generate the 96 trinucleotide mutation types (COSMIC-style)."""
    bases = ['A', 'C', 'G', 'T']  # Use uppercase to match the signature file
    mutations = ['C>A', 'C>G', 'C>T', 'T>A', 'T>C', 'T>G']
    types = [f"{b1}[{mut}]{b2}" for mut in mutations for b1 in bases for b2 in bases]
    return types

In [11]:
# Load exposures
df_exposure = pd.read_csv(exposure_file, sep="\t")
df_exposure = df_exposure.rename(columns={'aliquot_id': 'sample_id', 'sample_id': 'icgc_specimen_id'})
# Load signatures and ensure the correct order
catalog = pd.read_csv(signature_file)
catalog = ensure_signature_order(catalog)
signames = catalog.columns[1:]  # Exclude the 'Type' column
# Filter exposures for the current sample

exposures = df_exposure.query('sample_id == @sample')[signames]

# Filter signatures for non-zero exposures
exposures = exposures.loc[:, (exposures > 0).values.flatten()]
signatures = catalog.loc[:, exposures.columns]

In [12]:
likelihood_df = tau.sig_likelihoods(sample, df=normalize_df, exposures=exposures, signatures= signatures,
                                    ref_genome_path='/n/data1/hms/dbmi/park/jbrew/ref/hg19_decoy/human_g1k_v37_decoy.fasta')

calling get_context_96
context df:
  chrom       vaf      pos  nalt  nref alt ref  purity  location  major_cn  \
0     1  0.588235  3219419    20    17   T   C    0.91  13219419       2.0   
1     1  0.454545  4696822    20    28   A   G    0.91  14696822       2.0   
2     1  0.195122  4928360     8    37   C   T    0.91  14928360       2.0   
3     1  0.403509  5100918    23    40   T   A    0.91  15100918       2.0   
4     1  0.133333  5929305     6    43   G   A    0.91  15929305       2.0   

   ...  clonal_cn_state dist_clon   dist_sub  mean_count  sd_count  \
0  ...          2.0_1.0  0.000000  16.015330    1.242215  0.538823   
1  ...          2.0_1.0  0.944432  15.639015    1.242215  0.538823   
2  ...          2.0_1.0  0.865069  13.437574    1.242215  0.538823   
3  ...          2.0_1.0  0.772291  15.429588    1.242215  0.538823   
4  ...          2.0_1.0  2.334322  11.650078    1.242215  0.538823   

   mean_count_sub  sd_count_sub    weight context index_spectrum  
0       

In [13]:
multiplicities = tau.process_likelihood_file(df=likelihood_df)